# Multi Feature LSTM Hyperparameter Tuning

Importing the necessary libraries.

In [74]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense,LSTM,Dropout
import keras_tuner

Reading the data.

In [75]:
data = pd.read_csv('Google_train_data.csv')
data["Close"]=pd.to_numeric(data.Close,errors='coerce')
data = data.dropna()

In [76]:
trainData = data.iloc[:,1:5].values

Scaling.

In [77]:
scaler = MinMaxScaler(feature_range=(0,1))
trainData = scaler.fit_transform(trainData)

Splitting into X and y training set.

In [78]:
X_train = []
y_train = []

for i in range (60,1149): #60 : timestep // 1149 : length of the data
    X_train.append(trainData[i-60:i]) 
    y_train.append(trainData[i,3:4])

X_train,y_train = np.array(X_train),np.array(y_train)
lenofData = len(X_train)

print(y_train.shape)

(1089, 1)


Getting validation set (standard is 80/20)

In [79]:
# Split data into training and validation sets (80% train, 20% validation)
split_ratio = 0.8  # 80% for training, 20% for validation
train_size = int(lenofData * split_ratio)


# Create the training set (80%) and validation set (20%)
X_train, X_val = X_train[:train_size], X_train[train_size:]
y_train, y_val = y_train[:train_size], y_train[train_size:]
X_val.shape

(218, 60, 4)

Adding batch size.

In [80]:
X_train = np.reshape(X_train,(X_train.shape[0],X_train.shape[1],4)) #adding the batch_size axis
X_train.shape

(871, 60, 4)

Hyper paramter tuning:

This is where the fun happens. Because we are using Keras for the LSTM, its more work to use skit-learn's GridSearch for hyperparamer tuning, so we are using

In [84]:
from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasClassifier

def build_model(hp):
    model = Sequential()
    # Tune the number of LSTM units
    units = hp.Int('units', min_value=50, max_value=300, step=50)
    model.add(LSTM(units, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))
    batch_sizes = hp.Choice('batch_size', [16, 32, 64, 128, 264])
    # Tune the dropout rate
    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.6, step=0.1)
    model.add(Dropout(dropout_rate))
    model.add(Dense(units=1))
    model.compile(
        optimizer='adam', #should we tune this as well?
        loss='mean_squared_error', #how do we decide between them?
        metrics=['mean_absolute_error']
    )
    return model

Tuning:

In [87]:
# Initialize the tuner
tuner = keras_tuner.RandomSearch(
    build_model,
    objective='val_mean_absolute_error',
    max_trials=20,  # Number of different hyperparameter combinations to try
    executions_per_trial=1,  # Number of models to train per combination
    #directory='my_dir', #Where it saves all the trials
    #project_name='lstm_tuning'
)

Reloading Tuner from ./untitled_project/tuner0.json


In [88]:
# Perform the search
tuner.search(
    X_train,
    y_train,
    epochs=10, #use less epochs for training to make it faster
    validation_data=(X_val, y_val))

# Retrieve the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best number of units: {best_hps.get('units')}")
print(f"Best dropout rate: {best_hps.get('dropout_rate')}")
print(f"Best batch size: {best_hps.get('batch_size')}")

Trial 20 Complete [00h 00m 11s]
val_mean_absolute_error: 0.022767119109630585

Best val_mean_absolute_error So Far: 0.021298736333847046
Total elapsed time: 00h 04m 56s
Best number of units: 300
Best dropout rate: 0.30000000000000004
Best batch size: 16


In [469]:
best_model = tuner.hypermodel.build(best_hps)
history = best_model.fit(
    X_train,
    y_train,
    epochs=60,  # Max possible epochs
    validation_data=(X_val, y_val),
    verbose = 2,
    #callbacks=[keras_tuner.callbacks.EarlyStopping(monitor='val_loss', patience=3)]
)

Epoch 1/60
28/28 - 2s - 72ms/step - accuracy: 0.4856 - loss: 0.0566 - val_accuracy: 0.0321 - val_loss: 0.0150
Epoch 2/60
28/28 - 1s - 25ms/step - accuracy: 0.6349 - loss: 0.0131 - val_accuracy: 0.3303 - val_loss: 0.0031
Epoch 3/60
28/28 - 1s - 25ms/step - accuracy: 0.6372 - loss: 0.0098 - val_accuracy: 0.6376 - val_loss: 0.0013
Epoch 4/60
28/28 - 1s - 25ms/step - accuracy: 0.6613 - loss: 0.0086 - val_accuracy: 0.6239 - val_loss: 0.0014
Epoch 5/60
28/28 - 1s - 27ms/step - accuracy: 0.6521 - loss: 0.0084 - val_accuracy: 0.0321 - val_loss: 0.0043
Epoch 6/60
28/28 - 1s - 26ms/step - accuracy: 0.6613 - loss: 0.0072 - val_accuracy: 0.0321 - val_loss: 0.0027
Epoch 7/60
28/28 - 1s - 27ms/step - accuracy: 0.6544 - loss: 0.0066 - val_accuracy: 0.0321 - val_loss: 0.0028
Epoch 8/60
28/28 - 1s - 26ms/step - accuracy: 0.6521 - loss: 0.0063 - val_accuracy: 0.6376 - val_loss: 0.0022
Epoch 9/60
28/28 - 1s - 27ms/step - accuracy: 0.6406 - loss: 0.0059 - val_accuracy: 0.6330 - val_loss: 0.0016
Epoch 10/6

Now I'm going to use the same hyperparameter formating to tune to new data set.

In [5]:
data = pd.read_csv('tesla.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2193 entries, 0 to 2192
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       2193 non-null   object 
 1   Open       2193 non-null   float64
 2   High       2193 non-null   float64
 3   Low        2193 non-null   float64
 4   Close      2193 non-null   float64
 5   Adj Close  2193 non-null   float64
 6   Volume     2193 non-null   int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 120.1+ KB


In [6]:
data = data.iloc[:,1:6].values #to numpyarray

In [7]:
splitValue = int(len(data) * 0.8)
trainingData, testingData = data[:splitValue], data[splitValue:]

In [8]:
scMulti = MinMaxScaler(feature_range=(0,1))
trainingData = scMulti.fit_transform(trainingData)

In [19]:
X_train = []
y_train = []

for i in range (60,1149): #60 : timestep // 1149 : length of the data #we can adjsut the timestep??
    X_train.append(trainingData[i-60:i]) 
    y_train.append(trainingData[i])

X_train,y_train = np.array(X_train),np.array(y_train)
print(X_train.shape)
print(y_train.shape)

lenofData = len(X_train)

(1089, 60, 5)
(1089, 5)


In [20]:
# Split data into training and validation sets (80% train, 20% validation)
split_ratio = 0.8  # 80% for training, 20% for validation
train_size = int(lenofData * split_ratio)


# Create the training set (80%) and validation set (20%)
X_train, X_val = X_train[:train_size], X_train[train_size:]
y_train, y_val = y_train[:train_size], y_train[train_size:]
X_val.shape

(218, 60, 5)

In [22]:
X_train = np.reshape(X_train,(X_train.shape[0],X_train.shape[1],5)) #adding the batch_size axis
X_train.shape

(871, 60, 5)

In [31]:
# Perform the search
tuner.search(
    X_train,
    y_train,
    epochs=10, #use less epochs for training to make it faster
    validation_data=(X_val, y_val))

# Retrieve the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best number of units: {best_hps.get('units')}")
print(f"Best dropout rate: {best_hps.get('dropout_rate')}")
print(f"Best batch size: {best_hps.get('batch_size')}")

Trial 20 Complete [00h 00m 06s]
val_accuracy: 0.31192660331726074

Best val_accuracy So Far: 0.34403669834136963
Total elapsed time: 00h 06m 35s
Best number of units: 50
Best dropout rate: 0.2
Best batch size: 64
